# 01 · Create the Database

This notebook generates a **synthetic customer dataset** for an advertising campaign and stores it in a SQLite database (`data/raw/advertising.db`).

The data is **intentionally dirty** to simulate a real-world CRM export: missing values, junk text, inconsistent categories, mixed number and date formats, outliers and duplicate records.

## 1. Imports

In [1]:
import sqlite3
from datetime import date, timedelta

import numpy as np
import pandas as pd

DB_PATH = "../data/raw/advertising.db"

rng = np.random.default_rng(42)
print("Libraries ready ✓")

Libraries ready ✓


## 2. Connect to (or create) the database

If the file does not exist, SQLite creates it automatically.

In [2]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
print(f"Database '{DB_PATH}' created ✓")

Database '../data/raw/advertising.db' created ✓


## 3. Create the `customers` table

Every column is stored as `TEXT` on purpose, to simulate poorly typed data coming from a CRM.

In [3]:
cursor.execute("DROP TABLE IF EXISTS customers")
cursor.execute("""
    CREATE TABLE customers (
        customer_id   TEXT,
        age           TEXT,
        gender        TEXT,
        country       TEXT,
        channel       TEXT,
        monthly_spend TEXT,
        clicks        TEXT,
        impressions   TEXT,
        conversions   TEXT,
        signup_date   TEXT
    )
""")
conn.commit()
print("Table 'customers' created ✓")

Table 'customers' created ✓


## 4. Generate dirty data

All the intentional "dirt" is introduced here, explained step by step.

In [4]:
n = 500

# --- Clean base columns ---
clean_age         = rng.integers(18, 65, n)
clean_spend       = rng.uniform(100, 5000, n).round(2)
clean_clicks      = rng.integers(0, 1000, n)
clean_impressions = rng.integers(500, 50000, n)
clean_conversions = rng.integers(0, 50, n)

**Dirt 1 — `age`:** missing values and junk text.

In [5]:
age = clean_age.astype(str).tolist()
for i in rng.choice(n, 40, replace=False):   # 40 missing values
    age[i] = ""
for i in rng.choice(n, 15, replace=False):   # 15 junk text values
    age[i] = rng.choice(["N/A", "unknown", "none", "??"])

**Dirt 2 — `gender`:** inconsistent spelling of the same category.

In [6]:
gender_options = (
    ["Male"]   * 40 + ["male"]   * 20 + ["M"] * 15 +
    ["Female"] * 40 + ["female"] * 20 + ["F"] * 15 +
    ["male "]  * 10 +   # trailing whitespace
    ["FEMALE"] * 10 +
    ["Other"] * 20 + [""] * 10 + ["N/A"] * 10 +
    ["Man"] * 15 + ["Woman"] * 15
)
gender = [rng.choice(gender_options) for _ in range(n)]

**Dirt 3 — `country`:** several variants of the same name.

In [7]:
country_options = (
    ["Mexico"] * 60 + ["mexico"] * 20 + ["MX"] * 15 + ["MEXICO"] * 10 +
    ["Colombia"] * 50 + ["colombia"] * 15 + ["COL"] * 10 +
    ["Argentina"] * 40 + ["argentina"] * 10 + ["ARG"] * 8 +
    ["Chile"] * 30 + ["chile"] * 10 + ["CL"] * 8 +
    ["Peru"] * 20 + ["Perú"] * 15 + ["PE"] * 6 +
    [""] * 12 + ["Other"] * 11
)
country = [rng.choice(country_options) for _ in range(n)]

**Dirt 4 — `channel`:** misspelled categories.

In [8]:
channel_options = (
    ["Instagram"] * 80 + ["instagram"] * 20 + ["INSTAGRAM"] * 10 +
    ["Facebook"]  * 70 + ["facebook"]  * 15 + ["FB"] * 10 +
    ["Google"]    * 60 + ["google"]    * 10 + ["Google Ads"] * 15 +
    ["TikTok"]    * 40 + ["tiktok"]    * 10 + ["Tik Tok"] * 8 +
    ["Email"]     * 30 + ["email"]     * 10 + ["e-mail"] * 5 +
    [""] * 17
)
channel = [rng.choice(channel_options) for _ in range(n)]

**Dirt 5 — `monthly_spend`:** mixed number formats.

In [9]:
monthly_spend = []
for i, s in enumerate(clean_spend):
    fmt = rng.integers(0, 6)
    if fmt == 0:
        monthly_spend.append(f"${s:,.2f}")     # "$1,234.56"
    elif fmt == 1:
        monthly_spend.append(f"{s:.2f} USD")   # "1234.56 USD"
    elif fmt == 2:
        monthly_spend.append(str(round(s)))    # "1235"  no decimals
    elif fmt == 3:
        monthly_spend.append("")               # missing
    elif fmt == 4:
        monthly_spend.append("N/A")            # text
    else:
        monthly_spend.append(f"{s:.2f}")       # clean

**Dirt 6 — `clicks`:** extreme outliers and missing values.

In [10]:
clicks = clean_clicks.astype(str).tolist()
for i in rng.choice(n, 8, replace=False):
    clicks[i] = str(rng.integers(50000, 200000))   # absurd outlier
for i in rng.choice(n, 20, replace=False):
    clicks[i] = ""                                  # missing

**Dirt 7 — `signup_date`:** five different date formats.

In [11]:
base_date = date(2023, 1, 1)
signup_dates = []
for _ in range(n):
    d = base_date + timedelta(days=int(rng.integers(0, 700)))
    fmt = rng.integers(0, 5)
    if fmt == 0:
        signup_dates.append(d.strftime("%d/%m/%Y"))    # 15/03/2023
    elif fmt == 1:
        signup_dates.append(d.strftime("%m-%d-%Y"))    # 03-15-2023
    elif fmt == 2:
        signup_dates.append(d.strftime("%Y/%m/%d"))    # 2023/03/15
    elif fmt == 3:
        signup_dates.append(d.strftime("%B %d, %Y"))   # March 15, 2023
    else:
        signup_dates.append(d.strftime("%Y-%m-%d"))    # 2023-03-15 (ISO)

**Dirt 8 — duplicates:** 50 repeated rows with a different ID (simulates double registration).

In [12]:
ids = [f"CUS_{i:04d}" for i in range(n)]
rows = list(zip(ids, age, gender, country, channel, monthly_spend,
                clicks,
                clean_impressions.astype(str),
                clean_conversions.astype(str),
                signup_dates))

dup_idx = rng.choice(n, 50, replace=False)
duplicates = []
for k, i in enumerate(dup_idx):
    row = list(rows[i])
    row[0] = f"CUS_DUP_{k:03d}"   # different ID, everything else identical
    duplicates.append(tuple(row))

all_rows = rows + duplicates

print(f"Total rows generated: {len(all_rows)} ({n} original + 50 duplicates)")

Total rows generated: 550 (500 original + 50 duplicates)


## 5. Insert into SQLite

In [13]:
cursor.executemany("""
    INSERT INTO customers VALUES (?,?,?,?,?,?,?,?,?,?)
""", all_rows)
conn.commit()
print(f"Rows inserted into the DB: {cursor.rowcount} ✓")

Rows inserted into the DB: 550 ✓


## 6. Check that the data is really dirty

In [14]:
df = pd.read_sql("SELECT * FROM customers LIMIT 20", conn)
print("First 20 rows:")
print(df.to_string())

First 20 rows:
   customer_id age  gender    country    channel monthly_spend clicks impressions conversions        signup_date
0     CUS_0000  22       F   Colombia  INSTAGRAM     $3,294.46    529       43967          34         2024-04-01
1     CUS_0001  54  Female       Peru     TikTok          4351    260        3572           4         2023-03-29
2     CUS_0002  48     Man      Chile     Google          2324    229       20177          37      July 26, 2023
3     CUS_0003  38              chile  Instagram     $1,314.41    322       23183           5         2024/10/01
4     CUS_0004  38     Man         MX  Instagram   1259.65 USD    597       39995           2         03-04-2023
5     CUS_0005  58  female       Peru     Google           N/A    242        6886           5         2023-10-12
6     CUS_0006  22   Other       Peru   Facebook          4101    914       21985          22         2024/10/21
7     CUS_0007  50  Female   colombia  Instagram       $615.86    479        8040

## 7. Summary of data quality issues

In [15]:
df_full = pd.read_sql("SELECT * FROM customers", conn)

print("=== ISSUE SUMMARY ===")
print(f"Total rows            : {len(df_full)}")
print(f"\nEmpty values per column:")
print((df_full == "").sum())
print(f"\n'N/A'-like values:")
print(df_full.isin(["N/A", "none", "unknown", "??"]).sum())
print(f"\nUnique values in 'country' : {df_full['country'].nunique()}")
print(f"Unique values in 'gender'  : {df_full['gender'].nunique()}")
print(f"Unique values in 'channel' : {df_full['channel'].nunique()}")
print(f"\nSamples of dirty 'monthly_spend':")
print(df_full['monthly_spend'].sample(10, random_state=1).tolist())
print(f"\nSamples of mixed 'signup_date':")
print(df_full['signup_date'].sample(8, random_state=2).tolist())

=== ISSUE SUMMARY ===
Total rows            : 550

Empty values per column:
customer_id       0
age              48
gender           27
country          24
channel          23
monthly_spend    92
clicks           21
impressions       0
conversions       0
signup_date       0
dtype: int64

'N/A'-like values:
customer_id       0
age              15
gender           23
country           0
channel           0
monthly_spend    86
clicks            0
impressions       0
conversions       0
signup_date       0
dtype: int64

Unique values in 'country' : 18
Unique values in 'gender'  : 13
Unique values in 'channel' : 16

Samples of dirty 'monthly_spend':
['N/A', '1531.09 USD', '$1,704.73', 'N/A', '4934', '2347', '', '2296.40', '$4,297.09', '1512.32']

Samples of mixed 'signup_date':
['07-08-2023', '2024-10-31', 'July 18, 2023', '29/03/2024', '2023/01/12', '2023/04/30', '2024-02-23', '2024/01/06']


In [16]:
conn.close()
print(f"✓ Database ready at '{DB_PATH}'")

✓ Database ready at '../data/raw/advertising.db'
